In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np
from pathlib import Path
#from sklearn.model_selection import test_train_split
#from sklearn.ensemble import RandomForestCalssifier
#from sklearn.impute import SimpleImputer
#from sklearn.metrics import accuracy_score, classification_report

In [ ]:
# USER INPUTS
sql_filename = "/home/mike/GitHub/ExternalData/YahooFinanceAnalysis/TestData_Do_Not_Delete/BaseDatasets/SP500_data_06062026_not_verified.db"

In [ ]:
# FIND ALL UNIQUE SECTORS
#select_query = """
#    SELECT DISTINCT Sector 
#    FROM portfolio_data;
#"""
select_query = """
    SELECT Sector, COUNT(*) AS Count 
    FROM portfolio_data
    GROUP BY Sector;
"""

conn = sqlite3.connect(sql_filename)
cursor = conn.cursor()
unique_sector_db = pd.read_sql_query(select_query, conn)
conn.close()

print(f"unique_sector_db is {unique_sector_db}\n")
# drop count from dataframe
unique_sector_db = unique_sector_db["Sector"]

# drop NaNs from dataframe
unique_sector_db = unique_sector_db.dropna()

# convert dataframe to list
unique_sector_list = unique_sector_db.values.tolist()
print(f"unique_sector_list is {unique_sector_list}")


In [77]:
# 1. Fetch ALL sector averages at the same time using GROUP BY
select_query = """
    SELECT Sector, Price_To_Sales, Price_To_Earnings, Price_To_Book, Cash_To_Debt_Over_Market_Cap, Operating_Margin
    FROM portfolio_data 
    WHERE Sector IS NOT NULL
"""

conn = sqlite3.connect(sql_filename)
metrics_df = pd.read_sql_query(select_query, conn)
conn.close()

# drop NaNs from dataframe 
#  SQL query captures this now
metrics_df = metrics_df.dropna()

#print(f"metrics_df is {metrics_df}")

# Set Sector as the index so it maps perfectly to the chart axes
metrics_df.set_index('Sector', inplace=True)
metrics_df = metrics_df.dropna()

# Compute Mean and Std dataframes
means_df = metrics_df.groupby('Sector').mean()
stds_df = metrics_df.groupby('Sector').std()

print(f"means_df is \n {means_df}\n \n----------------\n")
print(f"stds_df is \n {stds_df}")

means_df is 
                         Price_To_Sales  Price_To_Earnings  Price_To_Book  \
Sector                                                                     
Basic Materials               3.102581           5.054441     -18.377604   
Communication Services        4.982056         -21.859871      40.039825   
Consumer Cyclical             2.801526          39.961424       9.954031   
Consumer Defensive            2.294998          22.456421       0.301580   
Energy                        3.949165          24.048831       7.463536   
Financial Services            4.706823          21.466352      -3.940139   
Healthcare                    4.012918          26.662537       1.248015   
Industrials                   4.065454          38.182423     123.316754   
Real Estate                   7.756695         107.986054       1.765475   
Technology                   10.920600         -44.253155      12.185287   
Utilities                     3.040110          23.393641       1.838297  

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(14, 12), sharex=False)
axes = axes.flatten() # Flatten the 2D grid into an easily iterable 1D array

for ii, metric_ii in enumerate(means_df.columns):
    #print(f"metric_ii is {metric_ii}")
    #print(f"x is {means_df.index}")
    #print(f"y is {means_df[metric_ii]}")
    #print(f"y_err is {stds_df[metric_ii]}")
    axes[ii].errorbar(
        means_df.index,
        means_df[metric_ii],
        yerr = stds_df[metric_ii],
        fmt='o',            
        capsize=4,     
        color=[0, 0, 1],   
        markersize=6,       
        linewidth=1.2,
    )
    title_str = metric_ii.replace('_', ' ')
    axes[ii].set_title(title_str, fontsize=12, fontweight='bold')
    
    axes[ii].set_xlabel('Sector', fontsize=10)#, labelpad=8)
    axes[ii].set_ylabel('Value', fontsize=10, labelpad=8)
    axes[ii].tick_params(axis='x', rotation=45, labelsize=9)

In [ ]:
for ii, metric_ii in enumerate(metrics_list):  
    metrics_df_ii.errorbar(
        metric_ii, 
        metrics_df_ii, 
        yerr=metrics_df[metrics_list_pp[ii]], 
        fmt='o',        # 'o' creates a scatter point marker
        color='blue',   # Color of the points and lines
        ecolor='red',   # Color of the error bars specifically
        capsize=5,      # Width of the horizontal caps on error bars
        markersize=8    # Size of the scatter points
    )

# Hide the unused 6th subplot grid square
fig.delaxes(axes[5])

plt.suptitle("Cross-Sector Financial Metric Comparison", fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
# GET MEAN AND STD OF EACH METRIC
avg_metric = []
std_metric = []
for sector_ii in unique_sector_list: 
    #print(sector_ii)   
    select_compute_query = f"""
        SELECT 
            AVG(Price_To_Sales) AS Mean_P_S, 
            AVG(Price_To_Earnings) AS Mean_P_E, 
            AVG(Price_To_Book) AS Mean_P_B, 
            AVG(Cash_To_Debt_Over_Market_Cap) AS Mean_Cash_Debt, 
            AVG(Operating_Margin) AS Mean_Op_Margin
        FROM portfolio_data 
        WHERE Sector = "{sector_ii}"
    """
    #print(select_compute_query)
    conn = sqlite3.connect(sql_filename)
    cursor = conn.cursor()
    #cursor = conn.cursor()
    cursor.execute(select_compute_query)
    unique_sector_metrics_ii = cursor.fetchall()
    #unique_sector_metrics_ii = pd.read_sql_query(select_compute_query, conn)
    conn.close()
    print(unique_sector_metrics_ii)
    #print(select_query)

In [ ]:
# Create the scatter plot with error bars
plt.figure(figsize=(10, 6))
plt.errorbar(
    metrics, 
    means, 
    yerr=std_devs, 
    fmt='o',        # 'o' creates a scatter point marker
    color='blue',   # Color of the points and lines
    ecolor='red',   # Color of the error bars specifically
    capsize=5,      # Width of the horizontal caps on error bars
    markersize=8    # Size of the scatter points
)

# Formatting
plt.ylabel('Values')
plt.title('Scatter Plot with Vertical Error Bars')
plt.grid(axis='y', linestyle='--', alpha=0.5) # Adds a clean background grid

plt.show()

In [ ]:
# 1. Fetch ALL sector averages at the same time using GROUP BY
select_compute_query = """
    SELECT 
        Sector,
        AVG(Price_To_Sales) AS Avg_Price_To_Sales, 
        STDDEV(Price_To_Sales) AS Std_Price_To_Sales,
        AVG(Price_To_Earnings) AS Price_To_Earnings, 
        STDDEV(Price_To_Earnings) AS Std_Price_To_Earnings, 
        AVG(Price_To_Book) AS Avg_Price_To_Book, 
        STDDEV(Price_To_Book) AS Std_Price_To_Book, 
        AVG(Cash_To_Debt_Over_Market_Cap) AS Avg_Cash_To_Debt, 
        STDDEV(Cash_To_Debt_Over_Market_Cap) AS Std_Cash_To_Debt, 
        AVG(Operating_Margin) AS Avg_Operating_Margin,
        STDEV(Operating_Margin) AS Std_Operating_Margin    
    FROM portfolio_data 
    WHERE Sector IS NOT NULL
    GROUP BY Sector;
"""

conn = sqlite3.connect(sql_filename)
metrics_df = pd.read_sql_query(select_compute_query, conn)
conn.close()

# Set Sector as the index so it maps perfectly to the chart axes
metrics_df.set_index('Sector', inplace=True)

fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(14, 12), sharex=False)
axes = axes.flatten() # Flatten the 2D grid into an easily iterable 1D array

metrics_list = ['Avg_Price_To_Sales', 'Avg_Price_To_Earnings', 'Avg_Price_To_Book', 'Avg_Cash_To_Debt', 'Avg_Operating_Margin']
metrics_list_pp = ['Std_Price_To_Sales', 'Std_Price_To_Earnings', 'Std_Price_To_Book', 'Std_Cash_To_Debt', 'Std_Operating_Margin']

for ii, metric_ii in enumerate(metrics_list):
    # Sort the data so bars are clean and readable from highest to lowest
    metrics_df_ii = metrics_df[metric_ii]
    
    metrics_df_ii.errorbar(
        metric_ii, 
        metrics_df_ii, 
        yerr=metrics_df[metrics_list_pp[ii]], 
        fmt='o',        # 'o' creates a scatter point marker
        color='blue',   # Color of the points and lines
        ecolor='red',   # Color of the error bars specifically
        capsize=5,      # Width of the horizontal caps on error bars
        markersize=8    # Size of the scatter points
    )

# Hide the unused 6th subplot grid square
fig.delaxes(axes[5])

plt.suptitle("Cross-Sector Financial Metric Comparison", fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
## SUPPLEMENTAL BELOW... JUST USED TO GET STARTED

In [ ]:
# Get Database Data

# all columns in database: Symbol, Shares, Sector, Industry, Price, Total_Value, Market_Cap, Price_To_Sales, Price_To_Earnings, Price_To_Book, Cash_To_Debt_Over_Market_Cap, RnD_Over_Market_Cap, Operating_Margin, Share_Change_Value_Over_Market_Cap, Percent_Of_Portfolio
select_query = "SELECT Symbol, Sector, Market_Cap, Price_To_Sales, Price_To_Earnings, Price_To_Book, Cash_To_Debt_Over_Market_Cap, Operating_Margin from portfolio_data;"

# Read without pandas... I actually want directly in pandas 
#conn = sqlite3.connect(sql_filename)
#cursor = conn.cursor()
#cursor.execute(select_query)
#rows = cursor.fetchall()
#conn.close()

# read with pandas
database_df = pd.read_sql_query(query, conn)
conn.close()

In [ ]:
# Remove Data with NaNs ??

# Look at Data

# Define target and features
target = database_df["Sector"]
features = database_df["Price_To_Sales","Price_To_Earnings","Price_To_Book","Cash_To_Debt_Over_Market_Cap","Operating_Margin"]
